# Chapter 07: Advanced Text Generation Techniques and tools

In [1]:
%%capture
!pip install langchain openai langchain_openai transformers datasets accelerate sentence-transformers duckduckgo-search langchain_community

# Fix: Use GGML_CUDA=on instead of LLAMA_CUDA=on
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python==0.2.69 --force-reinstall --no-cache-dir


# Loading the model

In [2]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-08-25 04:07:29--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 3.171.171.65, 3.171.171.6, 3.171.171.104, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.65|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/a9cdcf6e9514941ea9e596583b3d3c44dd99359fb7dd57f322bb84a0adc12ad4?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1787634450&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2E5Y2RjZjZlOTUxNDk0MWVhOWU1OTY1ODNiM2QzYzQ0ZGQ5OTM1OWZiN2RkNTdmMzIyYmI4NGEwYWRjMTJhZDRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmWC1YZXQtQ2FzLVVpZD1wdWJsa

In [3]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from huggingface_hub import hf_hub_download
from langchain_community.llms import LlamaCpp

print("Downloading model from Hugging Face Hub (approx. 7.6GB)...")
# Safely download the precise quantized model file
model_local_path = hf_hub_download(
    repo_id="Microsoft/Phi-3-mini-4k-instruct-gguf",
    filename="Phi-3-mini-4k-instruct-fp16.gguf"
)
print(f"Download complete! File saved to: {model_local_path}")

print("Loading model into GPU VRAM...")
# Pass the verified download path directly into LlamaCpp
llm = LlamaCpp(
    model_path=model_local_path,
    n_gpu_layers=-1,   # Offloads all layers to your Colab GPU
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

print("🚀 Success! Your Phi-3 model is fully loaded and ready to use.")


Phi-3-mini-4k-instruct-fp16.gguf:   0%|          | 0.00/7.64G [00:00<?, ?B/s]

Download complete! File saved to: /root/.cache/huggingface/hub/models--Microsoft--Phi-3-mini-4k-instruct-gguf/snapshots/a64113399c2f6b8ad3e11c394733a2ddadaa7f33/Phi-3-mini-4k-instruct-fp16.gguf
Loading model into GPU VRAM...
🚀 Success! Your Phi-3 model is fully loaded and ready to use.


In [4]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

''

## Chains

In [5]:
from langchain_core.prompts import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [6]:
basic_chain = prompt | llm

In [7]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

' Hello Maarten, the answer to 1 + 1 is 2.'